# 03 — Build Final Curriculum Dataset

Merges per-university parsed CSVs into the final unified curriculum dataset.

**Inputs:**
- `data/processed/university/ysu_courses_parsed.csv`
- `data/processed/university/aua_courses_parsed.csv`
- `data/processed/university/nuaca_courses_parsed.csv`
- `data/processed/university/rau_courses_parsed.csv`

**Output:** `data/processed/university/final_curriculum_dataset.csv`

In [ ]:
import os, csv
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parent.parent
OUT_DIR = BASE / "data" / "processed" / "university"

## Schema

In [ ]:
SCHEMA = [
    "course_id",             # Sequential integer — stable reference key for this dataset
    "university",            # Full official English university name
    "program_name",          # English program name
    "program_code",          # Internal code: YSU speciality code / AUA abbreviation
    "degree_level",          # Bachelor / Master / General
    "course_code",           # Module/course code from source (AUA: CS100, NUACA: 6.22ICS001, RAU: Б1.О.01)
    "course_name",           # Best available English name; original if no English available
    "course_name_original",  # Original-language name (Armenian or Russian) when different from course_name
    "credits",               # Credit hours as float; blank when unavailable
    "component",             # Curriculum component: General / Professional / Mandatory / Elective / Other
    "semester",              # Semester number (NUACA only; blank for YSU, AUA, RAU)
    "description",           # Course description text (AUA only)
    "prerequisites",         # Prerequisites (AUA only)
    "assessment",            # Assessment type: Exam / Test (NUACA only)
    "academic_year",         # Academic year string (YSU only)
    "source_url",            # URL or file reference of source page
    "source_language",       # Language of original source data: Armenian / English / Russian
    "notes",                 # Data quality flags; blank when no issues
]


def empty_row():
    return {col: "" for col in SCHEMA}


def normalize_credits(value):
    """Return credits as float string (e.g. '3.0') or blank string."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return ""
    try:
        return str(float(value))
    except (ValueError, TypeError):
        return str(value).strip()

## Process YSU

In [ ]:
def process_ysu(log):
    df = pd.read_csv(OUT_DIR / "ysu_courses_parsed.csv")
    log.append(f"  YSU raw rows loaded: {len(df)}")

    # CLEAN 1: Remove exact duplicates (all columns).
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    log.append(f"  YSU exact duplicates removed: {removed} (within-program scraping artefacts)")
    log.append(f"  YSU rows after deduplication: {len(df)}")

    rows_out = []
    component_map = {
        "General educational component":      "General",
        "Professional educational component": "Professional",
        "Other educational modules":          "Other",
    }

    for _, r in df.iterrows():
        row = empty_row()
        row["university"]    = "Yerevan State University"
        row["program_name"]  = str(r.get("program_name", "")).strip()

        speciality = str(r.get("speciality", "")).strip()
        code_part  = speciality.split(" - ")[0].strip() if " - " in speciality else speciality
        row["program_code"]  = code_part
        row["degree_level"]  = str(r.get("degree_type", "")).strip()

        course_orig = str(r.get("course_name_original", "")).strip()
        row["course_name"]          = course_orig
        row["course_name_original"] = ""
        row["course_code"]          = str(r.get("chair_code", "")).strip()
        row["credits"]              = normalize_credits(r.get("credits"))
        row["component"]            = component_map.get(str(r.get("component", "")).strip(),
                                                        str(r.get("component", "")).strip())
        row["academic_year"]        = str(r.get("academic_year", "")).strip()
        row["source_url"]           = str(r.get("source_url", "")).strip()
        row["source_language"]      = "Armenian"

        flags = []
        if course_orig == "---":
            flags.append("placeholder_name: course listed in source without a title (elective slot)")
        if row["credits"] == "0.0":
            flags.append("zero_credits: source reports 0 credits (may be non-credit activity)")
        row["notes"] = "; ".join(flags)

        rows_out.append(row)

    placeholder_count = sum(1 for r in rows_out if "placeholder_name" in r["notes"])
    zero_credit_count = sum(1 for r in rows_out if "zero_credits" in r["notes"])
    log.append(f"  YSU placeholder names ('---') flagged: {placeholder_count} (Blockchain elective slots — kept)")
    log.append(f"  YSU zero-credit courses flagged: {zero_credit_count} (kept; likely non-credit activities)")
    log.append(f"  YSU final rows: {len(rows_out)}")
    return rows_out

## Process AUA

In [ ]:
def process_aua(log):
    df = pd.read_csv(OUT_DIR / "aua_courses_parsed.csv")
    log.append(f"  AUA raw rows loaded: {len(df)}")
    log.append("  AUA duplicates: 0 exact duplicates found; same-name courses across programs are distinct")

    rows_out = []
    for _, r in df.iterrows():
        row = empty_row()
        row["university"]           = "American University of Armenia"
        row["program_name"]         = str(r.get("program_name", "")).strip()
        row["program_code"]         = str(r.get("program", "")).strip()
        row["degree_level"]         = str(r.get("degree_type", "")).strip()
        row["course_code"]          = str(r.get("course_code", "")).strip()
        row["course_name"]          = str(r.get("title", "")).strip()
        row["course_name_original"] = ""
        row["credits"]              = normalize_credits(r.get("credits"))

        desc = r.get("description", "")
        row["description"]    = str(desc).strip() if not pd.isna(desc) else ""

        prereq = r.get("prerequisites", "")
        row["prerequisites"]  = str(prereq).strip() if not pd.isna(prereq) else ""

        row["source_url"]      = "https://cse.aua.am/courses/"
        row["source_language"] = "English"

        rows_out.append(row)

    missing_desc  = sum(1 for r in rows_out if not r["description"])
    missing_prereq = sum(1 for r in rows_out if not r["prerequisites"])
    log.append(f"  AUA missing descriptions: {missing_desc} (kept as blank — no data in source)")
    log.append(f"  AUA missing prerequisites: {missing_prereq} (blank = no prerequisites or not listed)")
    log.append(f"  AUA corequisites: omitted from schema (6/249 courses; recoverable from source if needed)")
    log.append(f"  AUA final rows: {len(rows_out)}")
    return rows_out

## Process NUACA

In [ ]:
def process_nuaca(log):
    df = pd.read_csv(OUT_DIR / "nuaca_courses_parsed.csv", encoding="utf-8-sig")
    log.append(f"  NUACA raw rows loaded: {len(df)}")
    log.append("  NUACA BOM marker handled via utf-8-sig encoding")

    before_exam = (df["assessment"] == "Exam.").sum()
    df["assessment"] = df["assessment"].str.strip().str.rstrip(".")
    log.append(f"  NUACA assessment normalized: {before_exam} 'Exam.' → 'Exam'")

    rows_out = []
    for _, r in df.iterrows():
        row = empty_row()
        row["university"]    = "National University of Architecture and Construction of Armenia"
        row["program_name"]  = str(r.get("program_name", "")).strip()
        row["degree_level"]  = str(r.get("degree_type", "")).strip()

        module_code = r.get("module_code", "")
        row["course_code"]   = str(module_code).strip() if not pd.isna(module_code) else ""
        row["course_name"]   = str(r.get("course_name", "")).strip()
        row["credits"]       = normalize_credits(r.get("credits"))
        row["assessment"]    = str(r.get("assessment", "")).strip()

        sem = r.get("semester", "")
        row["semester"]      = str(int(sem)) if not pd.isna(sem) else ""
        row["source_url"]    = str(r.get("source_url", "")).strip()
        row["source_language"] = "English"

        flags = []
        if not row["course_code"] and not row["credits"]:
            flags.append("non_credit_activity: no module code or credits (Physical Training entries)")
        row["notes"] = "; ".join(flags)

        rows_out.append(row)

    flagged = sum(1 for r in rows_out if r["notes"])
    log.append(f"  NUACA non-credit activities flagged: {flagged} (Physical Training — kept)")
    log.append(f"  NUACA final rows: {len(rows_out)}")
    return rows_out

## Process RAU

In [ ]:
def process_rau(log):
    df = pd.read_csv(OUT_DIR / "rau_courses_parsed.csv")
    log.append(f"  RAU raw rows loaded: {len(df)}")
    log.append("  RAU duplicates: 0 exact duplicates; 'Computer Science Specialization' (3 rows) are distinct tracks")

    comp_map = {
        "Mandatory":             "Mandatory",
        "Elective (formative)":  "Elective",
        "Elective (by choice)":  "Elective",
    }

    rows_out = []
    for _, r in df.iterrows():
        row = empty_row()
        row["university"]    = "Russian-Armenian University"
        row["program_name"]  = str(r.get("program_name", "")).strip()
        row["degree_level"]  = str(r.get("degree_type", "")).strip()
        row["course_code"]   = str(r.get("course_code", "")).strip()

        row["course_name"]          = str(r.get("course_name_english", "")).strip()
        row["course_name_original"] = str(r.get("course_name_russian", "")).strip()
        row["credits"]              = normalize_credits(r.get("credits"))

        comp_raw = str(r.get("component", "")).strip()
        row["component"]     = comp_map.get(comp_raw, comp_raw)
        row["source_url"]    = str(r.get("source", "")).strip()
        row["source_language"] = "Russian"

        flags = []
        if not row["credits"]:
            flags.append("missing_credits: credit value absent in PDF source (Probability Theory and Math Statistics)")
        row["notes"] = "; ".join(flags)

        rows_out.append(row)

    missing_cr = sum(1 for r in rows_out if "missing_credits" in r["notes"])
    log.append(f"  RAU missing credits: {missing_cr} row (kept as blank — absent in PDF source)")
    log.append(f"  RAU final rows: {len(rows_out)}")
    return rows_out

## Merge, assign IDs, and save

In [ ]:
log = ["=== BUILD LOG: final_curriculum_dataset.csv ===", ""]

log.append("--- YSU ---")
ysu   = process_ysu(log)
log.append("")

log.append("--- AUA ---")
aua   = process_aua(log)
log.append("")

log.append("--- NUACA ---")
nuaca = process_nuaca(log)
log.append("")

log.append("--- RAU ---")
rau   = process_rau(log)
log.append("")

all_rows = ysu + aua + nuaca + rau

# Assign sequential course_id
for i, row in enumerate(all_rows, start=1):
    row["course_id"] = str(i)

total = len(all_rows)
log.append(f"--- TOTALS ---")
log.append(f"  YSU:   {len(ysu)}")
log.append(f"  AUA:   {len(aua)}")
log.append(f"  NUACA: {len(nuaca)}")
log.append(f"  RAU:   {len(rau)}")
log.append(f"  TOTAL: {total}")

# Write CSV
out_path = OUT_DIR / "final_curriculum_dataset.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=SCHEMA)
    writer.writeheader()
    writer.writerows(all_rows)

log.append(f"\nWritten: {out_path}")

## Sanity check

In [ ]:
# Sanity checks
df_out = pd.read_csv(out_path)
assert len(df_out) == total, f"Row count mismatch: {len(df_out)} vs {total}"

log.append("\n--- SANITY CHECK ---")
log.append("By university:")
for uni, cnt in df_out["university"].value_counts().items():
    log.append(f"  {cnt:>5}  {uni}")
log.append("By degree_level:")
for deg, cnt in df_out["degree_level"].value_counts().items():
    log.append(f"  {cnt:>5}  {deg!r}")
log.append("By source_language:")
for lang, cnt in df_out["source_language"].value_counts().items():
    log.append(f"  {cnt:>5}  {lang}")
log.append(f"Rows with notes (flagged): {df_out['notes'].fillna('').ne('').sum()}")
log.append(f"Missing credits: {df_out['credits'].eq('').sum()} / {total}")
log.append(f"Missing course_name: {df_out['course_name'].eq('').sum()} / {total}")

print("\n".join(log))
print("\nDone.")